# One resource for each HTTP call

Keep the action unchanged. Replace the ready resource with a factory and a finalizer.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [1]:
import asyncio
from fastapi.testclient import TestClient
from pydantic import Field
from aoa.action_machine.auth import GuestRole, NoAuthCoordinator
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.connection import connection
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.resources.per_call_connection import PerCallConnection
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.fastapi import FastApiAdapter

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [2]:
class CounterDomain(BaseDomain):
    """Group this example's resource and action."""
    name = "counter"
    description = "Counter example"

class EmptyParams(BaseParams):
    """No input fields are needed for this case."""

@meta(description="Count calls in memory", domain=CounterDomain)
class CounterResource(BaseResource):
    """Keep a count and record lifecycle events."""
    def __init__(self, events):
        """Start with an empty count."""
        self.count = 0
        self.events = events
    def get_wrapper_class(self):
        """Allow direct use of this in-memory counter."""
        return None
    async def open(self):
        """Record opening without resetting the count."""
        self.events.append("open")
    async def close(self):
        """Record cleanup without acquiring external handles."""
        self.events.append("close")

class CountResult(BaseResult):
    """Expose the count reached by one call."""
    count: int = Field(description="Number of calls seen by this resource")

@meta(description="Increment the supplied counter", domain=CounterDomain)
@check_roles(GuestRole)
@connection(CounterResource, key="counter")
class CountAction(BaseAction[EmptyParams, CountResult]):
    """Increment the supplied resource once."""
    @summary_aspect("Count one call")
    async def count_summary(self, params, state, box, connections):
        """Return the updated count."""
        counter = connections["counter"]
        counter.count += 1
        return CountResult(count=counter.count)

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [3]:
events = []
declaration = PerCallConnection(lambda: CounterResource(events), lambda resource: resource.close())
app = (FastApiAdapter(machine=ActionProductMachine(loggers=[]),
         auth_coordinator=NoAuthCoordinator(context=Context()))
       .post("/count", CountAction, connections={"counter": declaration})
       .build())

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [4]:
with TestClient(app) as client:
    responses = [client.post("/count", json={}) for _ in range(2)]
assert [r.status_code for r in responses] == [200, 200]
assert [r.json()["count"] for r in responses] == [1, 1]
for response in responses:
    print(response.json())
assert events == ['open', 'close', 'open', 'close']
print("lifecycle:", events)

{'count': 1}
{'count': 1}
lifecycle: ['open', 'close', 'open', 'close']


## 5. Explain the result

Each factory call constructed a new counter. The scope opened it before the action and awaited cleanup before returning the response. Newness comes from CounterResource(events), not from the PerCallConnection name.